# Import packages

In [2]:
import pandas as pd
import numpy as np

from pyrosm import OSM
import requests

pd.set_option('display.max_columns', None)

In [3]:
import sys
from pathlib import Path

# Find project root directory
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "preprocessing").is_dir()
)

# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

# Import standardization functions
from preprocessing.standardize_types import (
    standardize_types,
    get_type_group
)

# Load data

In [4]:
df_real_1 = pd.read_csv(PROJECT_ROOT / "data" / "export_20261020150603.csv")
df_real_2 = pd.read_csv(PROJECT_ROOT / "data" / "export_20261020160458.csv")

print("Shape of df_real_1:", df_real_1.shape)
print("Shape of df_real_2:", df_real_2.shape)

Shape of df_real_1: (545312, 47)
Shape of df_real_2: (430605, 47)


In [5]:
df_raw = pd.concat([df_real_1, df_real_2], ignore_index=True)
print("Shape of combined df_raw:", df_raw.shape)
df_raw.head()

Shape of combined df_raw: (975917, 47)


,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,address_1,address_2,address_city,address_state,address_zip,address_country,address_type,latitude,longitude,msa_code,msa_name,year_built,bedrooms,full_baths,half_baths,square_feet,rent_amount,listing_deposit,property_listing_status,description,creation_time,deactivation_time,ss_raw_primary_number,ss_raw_street_predirection,ss_raw_street_name,ss_raw_street_suffix,ss_raw_street_postdirection,ss_raw_secondary_designator,ss_raw_secondary_number,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type
0,133822-8163780,133822,8163780.0,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28.0,28,6211 N Kenmore Ave,Apt 501,Chicago,IL,60660,US,Apartment,41.994980,-87.656840,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",NaN,1,1.0,NaN,800.0,1145.0,0.0,inactive,"HEAT/WATER INCLUDED, PETS WELCOME. PARKING - P...",2021-02-28 15:00:43,2021-04-03 21:02:34,6211,N,Kenmore,Ave,NaN,Apt,501,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230.0,Apartment
1,129521-8112041,129521,8112041.0,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296.0,218,730 N Milwaukee Ave,Apt 807,Chicago,IL,60642,US,Apartment,41.895200,-87.653970,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",NaN,1,1.0,NaN,909.0,2500.0,0.0,inactive,River West 1BR + Den! $1000 Bonus! Pets Okay! ...,2021-02-14 07:11:47,2021-03-16 15:16:23,730,N,Milwaukee,Ave,NaN,Apt,807,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919.0,Apartment
2,123813-8066896,123813,8066896.0,14582015,16668829,279329,21st Street Lofts,3L Living,137.0,137,61 E 21st St,Unit 433,Chicago,IL,60616,US,Apartment,41.853916,-87.624672,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",2017.0,1,1.0,NaN,572.0,1699.0,NaN,inactive,Discover Modern Living at 21st Street Lofts in...,2025-05-15 13:35:48,2025-05-20 20:05:40,61,E,21st,St,NaN,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
3,123813-8066896,123813,8066896.0,14582015,19878436,279329,21st Street Lofts,3L Living,137.0,137,61 E 21st St,Unit 433,Chicago,IL,60616,US,Apartment,41.853916,-87.624672,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",2017.0,1,1.0,NaN,572.0,1705.0,NaN,inactive,Discover Modern Living at 21st Street Lofts in...,2026-05-18 15:15:54,2026-06-08 18:47:46,61,E,21st,St,NaN,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
4,130630-8121250,130630,8121250.0,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54.0,54,2420 N Kedzie Blvd,Apt 206,Chicago,IL,60647,US,Apartment,41.925404,-87.708099,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",1929.0,0,1.0,NaN,400.0,1450.0,NaN,inactive,"Features hardwood floors, renovated kitchen wi...",2023-06-21 01:50:14,2023-07-11 12:09:18,2420,N,Kedzie,Blvd,NaN,Apt,206,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523.0,Apartment


# Create geospatial features

In [ ]:
df_geo_input = df_raw.copy()
df_geo_input.head()

,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,address_1,address_2,address_city,address_state,address_zip,address_country,address_type,latitude,longitude,msa_code,msa_name,year_built,bedrooms,full_baths,half_baths,square_feet,rent_amount,listing_deposit,property_listing_status,description,creation_time,deactivation_time,ss_raw_primary_number,ss_raw_street_predirection,ss_raw_street_name,ss_raw_street_suffix,ss_raw_street_postdirection,ss_raw_secondary_designator,ss_raw_secondary_number,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type
0,133822-8163780,133822,8163780.0,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28.0,28,6211 N Kenmore Ave,Apt 501,Chicago,IL,60660,US,Apartment,41.994980,-87.656840,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",NaN,1,1.0,NaN,800.0,1145.0,0.0,inactive,"HEAT/WATER INCLUDED, PETS WELCOME. PARKING - P...",2021-02-28 15:00:43,2021-04-03 21:02:34,6211,N,Kenmore,Ave,<NA>,Apt,501,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230.0,Apartment
1,129521-8112041,129521,8112041.0,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296.0,218,730 N Milwaukee Ave,Apt 807,Chicago,IL,60642,US,Apartment,41.895200,-87.653970,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",NaN,1,1.0,NaN,909.0,2500.0,0.0,inactive,River West 1BR + Den! $1000 Bonus! Pets Okay! ...,2021-02-14 07:11:47,2021-03-16 15:16:23,730,N,Milwaukee,Ave,<NA>,Apt,807,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919.0,Apartment
2,123813-8066896,123813,8066896.0,14582015,16668829,279329,21st Street Lofts,3L Living,137.0,137,61 E 21st St,Unit 433,Chicago,IL,60616,US,Apartment,41.853916,-87.624672,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",2017.0,1,1.0,NaN,572.0,1699.0,NaN,inactive,Discover Modern Living at 21st Street Lofts in...,2025-05-15 13:35:48,2025-05-20 20:05:40,61,E,21st,St,<NA>,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
3,123813-8066896,123813,8066896.0,14582015,19878436,279329,21st Street Lofts,3L Living,137.0,137,61 E 21st St,Unit 433,Chicago,IL,60616,US,Apartment,41.853916,-87.624672,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",2017.0,1,1.0,NaN,572.0,1705.0,NaN,inactive,Discover Modern Living at 21st Street Lofts in...,2026-05-18 15:15:54,2026-06-08 18:47:46,61,E,21st,St,<NA>,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610.0,Apartment
4,130630-8121250,130630,8121250.0,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54.0,54,2420 N Kedzie Blvd,Apt 206,Chicago,IL,60647,US,Apartment,41.925404,-87.708099,16980,"Chicago-Joliet-Naperville, IL-IN-WI MSA",1929.0,0,1.0,NaN,400.0,1450.0,NaN,inactive,"Features hardwood floors, renovated kitchen wi...",2023-06-21 01:50:14,2023-07-11 12:09:18,2420,N,Kedzie,Blvd,<NA>,Apt,206,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523.0,Apartment


In [34]:
print("Initial shape:", df_geo_input.shape)
df_geo_input = df_geo_input[['dwellsy_address_id', 'address_city', 'latitude', 'longitude']].drop_duplicates()
print("Final shape:", df_geo_input.shape)
df_geo_input.head()

Initial shape: (975917, 47)
Final shape: (263926, 4)


,dwellsy_address_id,address_city,latitude,longitude
0,133822-8163780,Chicago,41.994980,-87.656840
1,129521-8112041,Chicago,41.895200,-87.653970
2,123813-8066896,Chicago,41.853916,-87.624672
4,130630-8121250,Chicago,41.925404,-87.708099
7,118522-7969432,Evanston,42.041416,-87.683342


In [38]:
CATEGORY_TAGS = {
    "hospitals": {
        "amenity": ["hospital"]
    },
    "pharmacies": {
        "amenity": ["pharmacy"],
    },
    "supermarkets": {
        "shop": ["supermarket"]
    },
    "convenience_stores": {
        "shop": ["convenience"]
    },
    "restaurants": {
        "amenity": ["restaurant", "fast_food"]
    },
    "cafes": {
        "amenity": ["cafe"]
    },
    "banks": {
        "amenity": ["bank"]
    },
    "atms": {
        "amenity": ["atm"]
    },
    "parks": {
        "leisure": ["park", "playground"]
    },
    "public_transport": {
        "highway": ["bus_stop"],
        "railway": ["station", "halt"],
        "amenity": ["bus_station"]
    }
}


In [ ]:
# Configuration
SEARCH_RADII = [500, 1000, 1500]  # in meters
EARTH_R = 6_371_000

OSM_DIR = PROJECT_ROOT / "data" / "osm"
OSM_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://download.geofabrik.de/north-america/us"

AREAS = {
    "chicago": {
        "file": "illinois-latest.osm.pbf",
        "bbox": [-88.5, 41.4, -87.2, 42.5]
    },
    "dfw": {
        "file": "texas-latest.osm.pbf",
        "bbox": [-98.0, 32.2, -96.3, 33.4]
    },
    "dc": {
        "file": "district-of-columbia-latest.osm.pbf",
        "bbox": [-77.6, 38.6, -76.8, 39.2]
    },
}

In [ ]:
def download_pbf(filename):

    # Define the local file path
    path = OSM_DIR / filename

    # Skip download if the file already exists
    if path.exists():
        print(f"File already exists: {filename}")
        return path

    # Build the download URL
    url = f"{BASE_URL}/{filename}"

    print(f"Downloading {url} ...")

    # Use a temporary file to avoid keeping incomplete downloads
    tmp = path.with_suffix(".part")

    with requests.get(url, stream=True, timeout=60) as response:

        # Raise an error if the download request fails
        response.raise_for_status()

        # Download the file in chunks
        with open(tmp, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    # Rename the temporary file after a successful download
    tmp.rename(path)

    print(f"Saved to: {path}")

    return path

# Path local (sin descargar)
for area in AREAS.values():
    area["path"] = OSM_DIR / area["file"]

for area in AREAS.values():
    download_pbf(area["file"])

In [41]:
# Extract POI tags
OSM_FILTER = {}

for tags in CATEGORY_TAGS.values():
    for key, values in tags.items():
        OSM_FILTER.setdefault(key, set()).update(values)

OSM_FILTER = {
    key: sorted(values)
    for key, values in OSM_FILTER.items()
}


# Calculate geographic distances
def distance(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    a = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin((lon2 - lon1) / 2) ** 2
    )

    return EARTH_R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


# Generate geospatial features
def generate_features(lat, lon, category_pois):
    features = {}

    for category in CATEGORY_TAGS:
        poi_coords = category_pois[category]

        if len(poi_coords) > 0:
            distances = distance(
                lat,
                lon,
                poi_coords[:, 0],
                poi_coords[:, 1]
            )
        else:
            distances = np.array([])

        for radius in SEARCH_RADII:
            features[f"{category}_{radius}m"] = int(
                np.sum(distances <= radius)
            )

        features[f"nearest_{category}_m"] = (
            distances.min() if len(distances) > 0 else np.nan
        )

    return features

In [42]:
# Initialize feature DataFrame
feature_columns = []

for category in CATEGORY_TAGS:
    for radius in SEARCH_RADII:
        feature_columns.append(f"{category}_{radius}m")

    feature_columns.append(f"nearest_{category}_m")


features_df = pd.DataFrame(
    np.nan,
    index=df_geo_input.index,
    columns=feature_columns
)


# Process each geographic area
for area_name, area in AREAS.items():
    print(f"\nProcessing area: {area_name}")

    west, south, east, north = area["bbox"]

    area_mask = (
        df_geo_input["longitude"].between(west, east)
        & df_geo_input["latitude"].between(south, north)
    )

    area_locations = df_geo_input.loc[area_mask]

    if area_locations.empty:
        print("No locations found in this area.")
        continue

    print(f"Locations to process: {len(area_locations)}")

    osm = OSM(
        str(area["path"]),
        bounding_box=area["bbox"],
        complete_relations=True
    )

    pois = osm.get_pois(custom_filter=OSM_FILTER)

    print(f"POIs extracted: {len(pois)}")

    pois = pois[pois.geometry.notna()].copy()

    # Project geometries to a suitable coordinate system
    pois_projected = pois.to_crs(pois.estimate_utm_crs())

    # Calculate centroids
    poi_centroids = pois_projected.geometry.centroid

    # Convert centroids back to geographic coordinates
    poi_centroids = poi_centroids.to_crs(pois.crs)

    pois["poi_lat"] = poi_centroids.y
    pois["poi_lon"] = poi_centroids.x

    category_pois = {}

    for category, tags in CATEGORY_TAGS.items():
        category_mask = pd.Series(False, index=pois.index)

        for key, values in tags.items():
            if key in pois.columns:
                category_mask |= pois[key].isin(values)

        category_data = pois.loc[category_mask]

        category_pois[category] = category_data[
            ["poi_lat", "poi_lon"]
        ].to_numpy()

    for idx, row in area_locations.iterrows():
        location_features = generate_features(
            row["latitude"],
            row["longitude"],
            category_pois
        )

        for feature, value in location_features.items():
            features_df.loc[idx, feature] = value

    print(f"Finished processing: {area_name}")


# Combine original data and geospatial features
df_geo_feats = pd.concat(
    [df_geo_input, features_df],
    axis=1
)

print("Final shape:", df_geo_feats.shape)


Processing area: chicago
Locations to process: 143911
POIs extracted: 48344
Finished processing: chicago

Processing area: dfw
No locations found in this area.

Processing area: dc
Locations to process: 102046
POIs extracted: 6680
Finished processing: dc
Final shape: (263926, 44)


In [43]:
df_geo_feats

,dwellsy_address_id,address_city,latitude,longitude,hospitals_500m,hospitals_1000m,hospitals_3000m,nearest_hospitals_m,pharmacies_500m,pharmacies_1000m,pharmacies_3000m,nearest_pharmacies_m,supermarkets_500m,supermarkets_1000m,supermarkets_3000m,nearest_supermarkets_m,convenience_stores_500m,convenience_stores_1000m,convenience_stores_3000m,nearest_convenience_stores_m,restaurants_500m,restaurants_1000m,restaurants_3000m,nearest_restaurants_m,cafes_500m,cafes_1000m,cafes_3000m,nearest_cafes_m,banks_500m,banks_1000m,banks_3000m,nearest_banks_m,atms_500m,atms_1000m,atms_3000m,nearest_atms_m,parks_500m,parks_1000m,parks_3000m,nearest_parks_m,public_transport_500m,public_transport_1000m,public_transport_3000m,nearest_public_transport_m
0,133822-8163780,Chicago,41.994980,-87.656840,0.0,0.0,2.0,2700.075732,2.0,3.0,10.0,330.546980,2.0,3.0,20.0,275.818073,3.0,5.0,43.0,153.241915,18.0,44.0,226.0,142.394806,5.0,9.0,38.0,70.102583,2.0,6.0,25.0,285.361838,0.0,0.0,3.0,1841.005882,5.0,15.0,114.0,101.759048,18.0,43.0,363.0,116.271259
1,129521-8112041,Chicago,41.895200,-87.653970,0.0,0.0,6.0,2587.437341,1.0,1.0,25.0,93.731352,1.0,1.0,28.0,112.916486,2.0,4.0,86.0,114.469015,15.0,69.0,1259.0,133.164874,2.0,12.0,282.0,15.467792,1.0,4.0,109.0,31.627118,0.0,0.0,25.0,1258.049233,2.0,19.0,214.0,260.574322,20.0,66.0,902.0,44.990152
2,123813-8066896,Chicago,41.853916,-87.624672,0.0,1.0,1.0,834.926064,0.0,2.0,11.0,732.854244,0.0,2.0,16.0,723.389859,4.0,6.0,38.0,103.864072,24.0,80.0,386.0,76.389673,8.0,26.0,93.0,186.815968,1.0,9.0,43.0,295.673937,0.0,1.0,12.0,775.398387,8.0,31.0,162.0,362.245455,33.0,85.0,515.0,58.123987
4,130630-8121250,Chicago,41.925404,-87.708099,0.0,0.0,2.0,2743.833116,1.0,2.0,11.0,263.188731,0.0,4.0,33.0,671.971471,1.0,15.0,69.0,445.680922,27.0,97.0,373.0,94.738549,8.0,19.0,64.0,183.777795,0.0,4.0,26.0,642.341267,0.0,0.0,6.0,1670.464777,3.0,15.0,144.0,267.537048,26.0,91.0,644.0,66.105135
7,118522-7969432,Evanston,42.041416,-87.683342,0.0,0.0,2.0,1878.533853,0.0,1.0,6.0,851.642198,2.0,3.0,11.0,312.468765,1.0,3.0,26.0,341.388745,11.0,85.0,163.0,58.196489,6.0,24.0,29.0,209.365355,2.0,10.0,21.0,232.055072,0.0,2.0,6.0,656.425910,6.0,24.0,161.0,284.336656,19.0,69.0,301.0,134.500703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
975900,9008429-19071092960,Frederick,39.377945,-77.428230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
975904,6378724,Fredericksburg,38.276493,-77.532013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
975906,10357052-60114508633,Washington,38.938839,-77.028557,0.0,0.0,5.0,1550.559044,1.0,2.0,7.0,487.453959,2.0,5.0,22.0,319.640637,10.0,24.0,86.0,146.838068,30.0,71.0,395.0,337.614278,7.0,13.0,66.0,321.442904,0.0,3.0,29.0,743.261695,1.0,4.0,17.0,411.730265,12.0,40.0,224.0,179.065850,21.0,75.0,480.0,321.624789
975907,6960124-4574933597,Rockville,39.059666,-77.125900,0.0,0.0,0.0,13752.925990,0.0,0.0,0.0,11404.043608,0.0,0.0,0.0,11302.379989,0.0,0.0,0.0,11568.615242,0.0,0.0,0.0,11276.472270,0.0,0.0,0.0,11331.083315,0.0,0.0,0.0,11238.322883,0.0,0.0,0.0,11948.162687,0.0,0.0,0.0,10420.004771,0.0,0.0,0.0,10343.969796
